# TF-IDF：词频-逆文档频率
> 难度标签：高级 | 预计时长：10-20分钟 | 前置知识：无学习经验


> 所属模块：08_自然语言处理 | 源文件：TF_IDF.py | 核心功能：TF-IDF 加权、文本特征提取

## 概述

TF-IDF 是词袋模型的改进。TF（词频）衡量词在文档中的频率，IDF（逆文档频率）降低常见词的权重。两者相乘，让"对当前文档重要但在整个语料中不常见"的词获得高权重。

**导入必要的库**

In [ ]:
# 确保中文输出正常（Windows 环境兼容）
import sys
try:
    sys.stdout.reconfigure(encoding='utf-8')
except AttributeError:
    pass

import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer, TfidfTransformer
from sklearn.pipeline import make_pipeline
from sklearn.naive_bayes import MultinomialNB
from sklearn.model_selection import cross_val_score

## 数学原理

### 1. TF-IDF 的基本思想

TF-IDF（Term Frequency - Inverse Document Frequency）衡量一个词对文档的重要程度。核心直觉：一个词在当前文档中出现多（高 TF），但在其他文档中出现少（高 IDF），则该词具有很强的区分能力。

### 2. 词频（TF）

词 $t$ 在文档 $d$ 中的词频：

$$\text{TF}(t, d) = \frac{c(t, d)}{\sum_{t' \in d} c(t', d)}$$

其中 $c(t, d)$ 是词 $t$ 在文档 $d$ 中的出现次数，分母是文档 $d$ 的总词数。

### 3. 逆文档频率（IDF）

词 $t$ 的逆文档频率：

$$\text{IDF}(t) = \log \frac{N}{|\{d \in D : t \in d\}|}$$

其中 $N$ 是文档总数，$|\{d : t \in d\}|$ 是包含词 $t$ 的文档数。

- 常见词（如"的"、"是"）：出现在几乎所有文档中，IDF $\approx 0$
- 罕见词（如"量子计算"）：只出现在少数文档中，IDF 较大

### 4. TF-IDF 权重

$$\text{TF-IDF}(t, d) = \text{TF}(t, d) \times \text{IDF}(t)$$

**sklearn 的平滑版本**（默认）：

$$\text{IDF}(t) = \log \frac{N + 1}{|\{d : t \in d\}| + 1} + 1$$

加 1 避免除零，加 1 避免 IDF 为零。

### 5. L2 归一化

sklearn 的 `TfidfVectorizer` 在计算 TF-IDF 后还进行 **L2 归一化**：

$$\hat{x}_{t,d} = \frac{\text{TF-IDF}(t, d)}{\sqrt{\sum_{t'} \text{TF-IDF}(t', d)^2}}$$

使得每个文档向量的 L2 范数为 1，便于余弦相似度计算。

### 6. 从 BoW 到 TF-IDF 的变换

代码中展示了两种方式：

```python
# 方式一：直接计算 TF-IDF
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(corpus)

# 方式二：先 BoW 再变换
pipeline = make_pipeline(CountVectorizer(), TfidfTransformer())
```

数学上等价：$\text{BoW} \xrightarrow{\text{TF}} \text{TF} \xrightarrow{\times \text{IDF}} \text{TF-IDF} \xrightarrow{\text{L2 norm}} \hat{X}$

### 7. TF-IDF 的几何解释

TF-IDF 将文档映射到 $|V|$ 维空间中的单位球面上（L2 归一化后）。文档间的相似度用**余弦相似度**：

$$\cos(\theta) = \mathbf{x}_d \cdot \mathbf{x}_{d'} = \sum_t \hat{x}_{t,d} \cdot \hat{x}_{t,d'}$$

### 8. 代码与数学的对应

| 代码 | 数学含义 |
|------|----------|
| `TfidfVectorizer()` | 计算 TF-IDF 矩阵 $\hat{X} \in \mathbb{R}^{n \times |V|}$ |
| `fit_transform(corpus)` | 从语料库学习词汇表并计算 TF-IDF |
| `ngram_range=(1,2)` | 包含 unigram 和 bigram 特征 |
| `MultinomialNB` + TF-IDF | 朴素贝叶斯分类器，输入 TF-IDF 特征 |
| `make_pipeline(CountVectorizer(), TfidfTransformer())` | BoW → TF → IDF → L2 归一化 |
| `top_idx = row.argsort()[::-1][:3]` | 每个文档 TF-IDF 值最高的 3 个词 |

### 1. 示例语料库

运行 1. 示例语料库 的代码，观察算法在该环节的行为。

In [ ]:
corpus = [
    "机器学习是人工智能的基础技术",
    "深度学习是机器学习的子领域，用到神经网络",
    "自然语言处理是人工智能的应用之一",
    "计算机视觉用卷积神经网络处理图像",
# --- 继续 ---
    "强化学习通过奖励信号训练智能体",
    "Python是机器学习最常用的编程语言",
]

### 2. 中文分词

运行 2. 中文分词 的代码，观察算法在该环节的行为。

In [ ]:
try:
    import jieba
    corpus_cut = [" ".join(jieba.cut(doc)) for doc in corpus]
except ImportError:
    corpus_cut = [" ".join(doc) for doc in corpus]

print("=== 语料库 ===")
for i, doc in enumerate(corpus_cut):
    print(f"  文档{i+1}: {doc}")

### 3. TF-IDF 计算

运行 3. TF-IDF 计算 的代码，观察算法在该环节的行为。

In [ ]:
tfidf = TfidfVectorizer()
X = tfidf.fit_transform(corpus_cut)

print(f"\n=== TF-IDF 矩阵 ===")
print(f"形状: {X.shape}")
feature_names = tfidf.get_feature_names_out()

# 展示每个文档的 Top-3 词
for i in range(len(corpus_cut)):
    row = X[i].toarray().flatten()
    top_idx = row.argsort()[::-1][:3]
    print(f"  文档{i+1} Top-3: {[(feature_names[j], f'{row[j]:.3f}') for j in top_idx]}")

### 4. TF-IDF 公式

运行 4. TF-IDF 公式 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== TF-IDF 公式 ===")
print("TF(t,d) = 词 t 在文档 d 中的出现次数 / 文档 d 的总词数")
print("IDF(t) = log(总文档数 / 包含词 t 的文档数)")
print("TF-IDF(t,d) = TF(t,d) × IDF(t)")
print()
# --- 输出结果 ---
print("含义: 一个词在当前文档中频繁出现（TF高）且在其他文档中罕见（IDF高），则 TF-IDF 高")

# 手动计算一个词的 TF-IDF 验证
print("\n手动验证（'机器' 在文档 1 中）:")
# 找到'机器'的索引
machine_idx = list(feature_names).index("机器")
tf = X[0, machine_idx]  # sklearn 已归一化
print(f"  TF-IDF (sklearn): {tf:.4f}")

### 5. 参数对比

探索关键超参数对模型行为的影响。

In [ ]:
print("\n=== 参数对比 ===")

# sublinear_tf: 对 TF 取 log(1+TF)，缓解高频词主导
for sublinear in [False, True]:
    vec = TfidfVectorizer(sublinear_tf=sublinear)
    X_t = vec.fit_transform(corpus_cut)
    print(f"  sublinear_tf={sublinear}: 形状={X_t.shape}")

# max_df / min_df
vec_df = TfidfVectorizer(min_df=1, max_df=0.8)
X_df = vec_df.fit_transform(corpus_cut)
print(f"  min_df=1, max_df=0.8: 词汇={len(vec_df.vocabulary_)}")

# norm: L1 或 L2 归一化
for norm in ["l1", "l2", None]:
    vec_n = TfidfVectorizer(norm=norm)
    X_n = vec_n.fit_transform(corpus_cut)
    if norm:
        l2_norms = np.linalg.norm(X_n.toarray(), axis=1)[:3]
# --- 输出结果 ---
        print(f"  norm={norm}: 样本L2范数={l2_norms.round(4)}")

### 6. TF-IDF vs CountVectorizer

运行 6. TF-IDF vs CountVectorizer 的代码，观察算法在该环节的行为。

In [ ]:
print("\n=== TF-IDF vs 词袋模型 ===")
from sklearn.feature_extraction.text import CountVectorizer

docs_class = [
    "机器学习 深度学习 神经网络",
    "自然语言处理 文本 语义",
    "机器学习 算法 模型",
    "图像 卷积神经网络 视觉",
# --- 继续 ---
    "文本 分词 自然语言",
    "深度学习 图像 模型",
] * 5
labels = [0, 1, 0, 2, 1, 2] * 5

for name, vec_cls in [("CountVectorizer", CountVectorizer), ("TfidfVectorizer", TfidfVectorizer)]:
    pipe = make_pipeline(vec_cls(), MultinomialNB())
    scores = cross_val_score(pipe, docs_class, labels, cv=3, scoring="accuracy")
    print(f"  {name:>20} + NB: CV准确率={scores.mean():.4f}")

print("\n=== TF-IDF 要点 ===")
print("- 比词袋模型更好：惩罚在所有文档都出现的常见词")
print("- sublinear_tf=True: 常用设置，对 TF 取 log")
print("- 适合：文本分类、信息检索、关键词提取")
print("- 不考虑词序：仍然无法捕捉'学习机器'vs'机器学习'的区别")

## 关键代码解释

```python
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(max_features=5000, sublinear_tf=True)
X = tfidf.fit_transform(texts)
```

`sublinear_tf=True` 用 1+log(tf) 替代 tf，压缩高频词的影响。

## 注意事项

1. 仍然是稀疏高维向量
2. IDF 在训练集上计算，应用到测试集
3. 中文需要先分词

## 总结与延伸

以上代码展示了 **TF IDF** 的完整流程。

**解读要点**：
- 观察 **词汇表大小** 和 **向量维度** 是否合理
- 检查相似词查询结果是否符合语义直觉
- 关注分类任务中的 **TF-IDF 权重** 分布

**进一步思考**：尝试修改关键参数（如正则化强度、学习率、树深度等），观察结果如何变化。

### 延伸阅读

- **BM25**：TF-IDF 的改进版，搜索引擎的标准
- **TF-IDF + SVM**：经典的文本分类组合
- **Transformer 时代**：BERT 等预训练模型是否完全取代了 TF-IDF
